In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("heart_disease_uci.csv")
cleveland_df = df[df['dataset'] == 'Cleveland'].iloc[:304]

In [ ]:
# Define features and target
features = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
X = cleveland_df[features]
y = cleveland_df['num'].apply(lambda x: 0 if x == 0 else 1)

In [ ]:
# Handle missing values using Pandas directly
X = X.fillna(X.select_dtypes(include=['int64', 'float64']).mean())
X = X.fillna(X.select_dtypes(exclude=['int64', 'float64']).mode().iloc[0])

<ipython-input-14-b605a8734f2a>:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.fillna(X.select_dtypes(exclude=['int64', 'float64']).mode().iloc[0])


In [ ]:
# Check class distribution
print("Class Distribution (0 = No Disease, 1 = Disease):")
print(y.value_counts(normalize=True))

Class Distribution (0 = No Disease, 1 = Disease):
num
0    0.542763
1    0.457237
Name: proportion, dtype: float64


In [ ]:
# Encode categorical variables
le = LabelEncoder()
for col in ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope','ca','thal']:  # Add 'ca', 'thal' if used
    X[col] = le.fit_transform(X[col].astype(str))

In [ ]:
# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  # Stratify to balance classes
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 1. Logistic Regression with class weights
lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_pred)

In [ ]:
# 2. Random Forest with expanded tuning and class weights
rf_param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}
rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf, rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
rf_pred = rf_best.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Best Random Forest Params: {rf_grid.best_params_}")

Best Random Forest Params: {'class_weight': None, 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}


In [ ]:
# 3. SVM with expanded tuning and class weights
svm_param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto', 0.1],
    'class_weight': ['balanced', None]
}
svm = SVC(random_state=42)
svm_grid = GridSearchCV(svm, svm_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train, y_train)
svm_best = svm_grid.best_estimator_
svm_pred = svm_best.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_pred)
print(f"Best SVM Params: {svm_grid.best_params_}")

Best SVM Params: {'C': 100, 'class_weight': None, 'gamma': 'scale', 'kernel': 'linear'}


In [ ]:
# Results
print("\nModel Performance (Accuracy):")
print(f"Logistic Regression: {lr_accuracy:.4f}")
print(f"Random Forest: {rf_accuracy:.4f}")
print(f"SVM: {svm_accuracy:.4f}")


Model Performance (Accuracy):
Logistic Regression: 0.8852
Random Forest: 0.9180
SVM: 0.9344
